# Week 9 — 重构式无监督异常检测 (MVP v0.4 上)

**目标**：用 Transformer Encoder 学"正常"的内部规律，用重构误差检测异常。

**产出**：
- 合成数据上 AUC-PR ≥ 0.80
- Kaggle 数据上有监督 vs 无监督对比数字
- 简化版 Anomaly Transformer (Association Discrepancy) 消融


In [ ]:
# ── Bootstrap (Colab + local) ──────────────────────────────────────
# Env detect → Drive mount → Kaggle creds → reproducibility
import os, sys, pathlib, random

IN_COLAB = 'google.colab' in sys.modules
DRIVE_FOLDER = 'LLM/transformer-anomaly'  # edit if your Drive subfolder differs

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    PROJECT_ROOT = pathlib.Path(f'/content/drive/MyDrive/{DRIVE_FOLDER}')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for sub in ('data', 'models', 'checkpoints'):
        (PROJECT_ROOT / sub).mkdir(exist_ok=True)
    os.chdir(PROJECT_ROOT)

    # Kaggle credentials via Colab Secrets (left sidebar 🔑 icon).
    # Add KAGGLE_USERNAME and KAGGLE_KEY there; never upload kaggle.json.
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    except Exception as e:
        print(f'[bootstrap] Kaggle secrets not configured: {e}')
else:
    PROJECT_ROOT = pathlib.Path.cwd()

# Reproducibility
import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[bootstrap] env={"colab" if IN_COLAB else "local"}  device={device}')
print(f'[bootstrap] project_root={PROJECT_ROOT}')

## 0. 无监督异常检测三大范式 (理论热身)

| 范式 | 核心假设 | 典型模型 | 优势 | 弱点 |
|------|----------|----------|------|------|
| **重构 (reconstruction)** | 正常样本能被低维瓶颈/生成器精确还原，异常还原误差大 | AutoEncoder、VAE、MAE、Anomaly Transformer 重构分支 | 不需要标签；直观可解释 | 模型过强时会把异常也重构掉；对点异常不敏感 |
| **预测 (prediction / next-step)** | 正常序列下一步可预测，异常打断模式 | LSTM forecaster、GPT-style next token | 天然适配时序、在线推理 | 对长尾起点/多模态难；训练需设预测步长 |
| **对比 (contrastive)** | 正常样本之间有共同表征，异常远离聚类中心 | Deep SVDD、CPC、SimCLR + OC-SVM | 自监督、可学稳健表征 | 需要设计好的正/负样本增强 |

**选型经验**：
- 特征之间有强互相关（本周合成数据、支付多字段） → 重构
- 明显的时间自回归（价格、心跳） → 预测
- 特征空间已经很好、样本量大 → 对比

本周主攻**重构**，并用 Association Discrepancy 消融体验"预测 + 对比"的思想。


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, math, time
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')

torch.manual_seed(SEED); np.random.seed(SEED)
print('torch', torch.__version__, '| device', device)

## 1. 合成数据：5 个特征 + 3 类异常

**设计动机**：
- `amount`：主字段，`AR(1) + 正弦季节性 + 噪声`，模拟交易金额。
- 其余 4 个特征：2 个与 amount 正相关、2 个无关噪声 → 体现多元结构。
- 注入 3 类异常：
  1. **点异常 (point)**：单个 timestep amount 跳到 >6σ。
  2. **上下文异常 (contextual)**：amount 本身正常，但相关特征失联（相关性断裂）。
  3. **集体异常 (collective)**：连续 5-10 个 timestep 被一段不同规律的序列替换。

训练集：**100% 只含正常序列**（无监督假设）。
评估集：含 30% 异常，但标签"藏起来"，只用来计算 AUC-PR。


In [ ]:
def make_normal_sequence(T=64, F=5, rng=None):
    rng = rng or np.random.default_rng()
    t = np.arange(T)
    # amount: AR(1) + seasonal + noise
    amt = np.zeros(T)
    phi = 0.7
    for i in range(1, T):
        amt[i] = phi * amt[i-1] + rng.normal(0, 0.5)
    amt += 0.6 * np.sin(2 * np.pi * t / 16)
    # 4 其它特征
    f1 = 0.8 * amt + rng.normal(0, 0.3, T)   # 强相关
    f2 = 0.5 * amt + rng.normal(0, 0.4, T)   # 中相关
    f3 = rng.normal(0, 1, T)                  # 无关噪声
    f4 = 0.2 * np.cos(2 * np.pi * t / 8) + rng.normal(0, 0.3, T)  # 独立季节
    x = np.stack([amt, f1, f2, f3, f4], axis=-1).astype(np.float32)
    return x


def inject_anomaly(x, kind, rng):
    x = x.copy()
    T, F = x.shape
    if kind == 'point':
        idx = rng.integers(5, T-5)
        x[idx, 0] += rng.choice([-1, 1]) * rng.uniform(6, 10)
    elif kind == 'contextual':
        s = rng.integers(0, T - 10); e = s + 8
        # 打碎 amount 与其它字段的相关性：用随机置换的其它字段
        perm = rng.permutation(F - 1) + 1
        x[s:e, 1:] = x[s:e, perm]
    elif kind == 'collective':
        s = rng.integers(0, T - 12); e = s + rng.integers(6, 11)
        # 替换为不同规律的子序列
        sub = np.zeros_like(x[s:e])
        sub[:, 0] = 3.0 + 0.3 * np.arange(e - s) + rng.normal(0, 0.2, e - s)
        sub[:, 1:] = rng.normal(0, 0.5, (e - s, F - 1))
        x[s:e] = sub
    return x


def build_dataset(n_normal=4000, n_eval_normal=600, n_eval_anom=200, seed=42):
    rng = np.random.default_rng(seed)
    train = np.stack([make_normal_sequence(rng=rng) for _ in range(n_normal)])

    eval_xs, eval_y, eval_kind = [], [], []
    for _ in range(n_eval_normal):
        eval_xs.append(make_normal_sequence(rng=rng)); eval_y.append(0); eval_kind.append('normal')
    kinds = ['point', 'contextual', 'collective']
    for _ in range(n_eval_anom):
        k = kinds[rng.integers(3)]
        x = inject_anomaly(make_normal_sequence(rng=rng), k, rng)
        eval_xs.append(x); eval_y.append(1); eval_kind.append(k)
    eval_xs = np.stack(eval_xs); eval_y = np.array(eval_y); eval_kind = np.array(eval_kind)

    # standardize by train stats
    mu = train.mean((0, 1), keepdims=True); sd = train.std((0, 1), keepdims=True) + 1e-6
    train = (train - mu) / sd
    eval_xs = (eval_xs - mu) / sd
    return train, eval_xs, eval_y, eval_kind

train_x, eval_x, eval_y, eval_kind = build_dataset()
print('train', train_x.shape, '| eval', eval_x.shape, '| fraud rate', eval_y.mean().round(3))

In [ ]:
# 先看几条样本，确认异常"眼睛能看出来"
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
samples = {
    'normal': np.where(eval_kind == 'normal')[0][0],
    'point': np.where(eval_kind == 'point')[0][0],
    'contextual': np.where(eval_kind == 'contextual')[0][0],
    'collective': np.where(eval_kind == 'collective')[0][0],
}
for ax, (name, idx) in zip(axes.flat, samples.items()):
    for f, lab in enumerate(['amount', 'f1', 'f2', 'f3', 'f4']):
        ax.plot(eval_x[idx, :, f], label=lab, alpha=0.8 if f == 0 else 0.4)
    ax.set_title(f'{name} (idx={idx})'); ax.legend(fontsize=7)
plt.tight_layout()

## 2. 模型：重构式 Transformer Encoder

沿用 Week 7 的"自己写 Encoder"模式：
- `d_model=64, n_heads=4, n_layers=4, dim_ff=128`
- 可学习位置编码 (T=64)
- 无 causal mask（重构需要看全序列）
- **重构头**：`Linear(d_model, F_in)`，每个 timestep 还原原始输入

训练目标 = 对**全部 timestep** 的 MSE。推理时，对每条序列取 `max(MSE_t)` 或 `mean(MSE_t)` 作为异常分数。


In [ ]:
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)

    def forward(self, x, return_attn=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.h, self.dk).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        attn = attn.softmax(-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, D)
        out = self.o(out)
        if return_attn:
            return out, attn   # attn shape: (B, h, T, T)
        return out


class EncoderBlock(nn.Module):
    def __init__(self, d=64, h=4, ff=128, p=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.attn = MHA(d, h)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))
        self.drop = nn.Dropout(p)

    def forward(self, x, return_attn=False):
        if return_attn:
            a, w = self.attn(self.ln1(x), return_attn=True)
            x = x + self.drop(a)
            x = x + self.drop(self.ff(self.ln2(x)))
            return x, w
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x


class ReconTransformer(nn.Module):
    def __init__(self, n_feat, T=64, d=64, h=4, n_layers=4, ff=128, p=0.1):
        super().__init__()
        self.in_proj = nn.Linear(n_feat, d)
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        self.blocks = nn.ModuleList([EncoderBlock(d, h, ff, p) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_feat)

    def forward(self, x, return_attn=False):
        h = self.in_proj(x) + self.pos[:, : x.size(1)]
        attns = []
        for blk in self.blocks:
            if return_attn:
                h, w = blk(h, return_attn=True); attns.append(w)
            else:
                h = blk(h)
        h = self.ln_f(h)
        recon = self.head(h)
        if return_attn:
            return recon, attns
        return recon

model = ReconTransformer(n_feat=5).to(device)
print(sum(p.numel() for p in model.parameters()), 'params')

## 3. 训练（纯重构 20 epochs）

In [ ]:
def make_loader(x, bs=128, shuffle=True):
    return DataLoader(TensorDataset(torch.from_numpy(x)), batch_size=bs, shuffle=shuffle)

train_loader = make_loader(train_x)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)

losses = []
t0 = time.time()
for ep in range(20):
    model.train(); ep_loss = 0; n = 0
    for (xb,) in train_loader:
        xb = xb.to(device)
        pred = model(xb)
        loss = F.mse_loss(pred, xb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        ep_loss += loss.item() * xb.size(0); n += xb.size(0)
    sched.step()
    losses.append(ep_loss / n)
    if ep % 2 == 0 or ep == 19:
        print(f'ep {ep:02d}  loss {losses[-1]:.4f}  lr {opt.param_groups[0]["lr"]:.2e}  elapsed {time.time()-t0:.0f}s')

plt.figure(figsize=(7, 3)); plt.plot(losses, marker='o')
plt.title('Recon MSE (train)'); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.tight_layout()

## 4. 异常分数 + AUC-PR

In [ ]:
@torch.no_grad()
def score_sequences(model, x, reduce='max', bs=128):
    model.eval()
    out = []
    for i in range(0, len(x), bs):
        xb = torch.from_numpy(x[i:i+bs]).to(device)
        pred = model(xb)
        err = (pred - xb).pow(2).mean(-1)      # (B, T) per-timestep MSE
        if reduce == 'max':
            out.append(err.max(-1).values.cpu().numpy())
        else:
            out.append(err.mean(-1).cpu().numpy())
    return np.concatenate(out)

scores_max = score_sequences(model, eval_x, 'max')
scores_mean = score_sequences(model, eval_x, 'mean')

def report(y, s, name):
    ap = average_precision_score(y, s)
    auc = roc_auc_score(y, s)
    print(f'{name:20s}  AUC-PR {ap:.4f}  AUC-ROC {auc:.4f}')
    return ap, auc

ap_max, _ = report(eval_y, scores_max, 'score=max(MSE_t)')
ap_mean, _ = report(eval_y, scores_mean, 'score=mean(MSE_t)')
print('\n验收目标：AUC-PR ≥ 0.80')

In [ ]:
# 按异常类型拆解（哪一类最容易、哪一类最难？）
for k in ['point', 'contextual', 'collective']:
    mask = (eval_kind == 'normal') | (eval_kind == k)
    y_sub = (eval_kind[mask] == k).astype(int)
    s_sub = scores_max[mask]
    ap = average_precision_score(y_sub, s_sub)
    auc = roc_auc_score(y_sub, s_sub)
    print(f'{k:12s}  AUC-PR {ap:.4f}  AUC-ROC {auc:.4f}')

In [ ]:
# 分数分布直方图（直观感受能否分开）
fig, ax = plt.subplots(figsize=(8, 3))
for k, c in zip(['normal', 'point', 'contextual', 'collective'], ['C0', 'C3', 'C1', 'C2']):
    ax.hist(scores_max[eval_kind == k], bins=40, alpha=0.5, label=k, color=c)
ax.set_xlabel('score = max_t MSE'); ax.set_ylabel('count'); ax.legend()
ax.set_title('异常分数分布（max reduction）'); plt.tight_layout()

## 5. 简化版 Anomaly Transformer：Association Discrepancy (消融)

原论文核心观察：

- **Prior-association (P)**：异常点几乎只关注自己附近的 timestep（局部），所以用一个**可学的高斯核**在相对位置上作为"先验关联"。
- **Series-association (S)**：正常的 attention 权重会指向远处的相似模式（周期、趋势）。
- **Discrepancy**：KL(P || S) —— 正常样本上 S 发散（远跳），P 集中 → KL 大；异常样本上 S 也集中 → KL 小。
- 最终 anomaly score 结合重构误差 × e^{-KL}（简化版做直接加权），鼓励让正常样本的 P/S 分歧更大。

我们做最简版：
1. 每层学一个 `sigma_l`（共享所有 head），`P[i,j] = N(j-i; 0, sigma_l^2)` 归一化。
2. `S` 用最后一层 attention 的均值。
3. 每条序列的 AD score = `mean_{i,j} KL(P[i,j] || S[i,j])`（softmax 后）。
4. 联合分数：`recon + lam * (-AD)`（AD 越小越异常）。

目的是体验 idea，不追求复现原文。


In [ ]:
class AnomalyTransformerLite(nn.Module):
    """只保留最后一层 attention 的 P/S，供可视化与消融。"""
    def __init__(self, n_feat, T=64, d=64, h=4, n_layers=4, ff=128, p=0.1):
        super().__init__()
        self.T = T
        self.in_proj = nn.Linear(n_feat, d)
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        self.blocks = nn.ModuleList([EncoderBlock(d, h, ff, p) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_feat)
        # learnable log-sigma per layer (shared over heads for simplicity)
        self.log_sigma = nn.Parameter(torch.zeros(n_layers))
        # precompute relative-position matrix
        idx = torch.arange(T)
        self.register_buffer('rel', (idx[None, :] - idx[:, None]).float())

    def forward(self, x, return_AD=False):
        h = self.in_proj(x) + self.pos[:, : x.size(1)]
        last_attn = None
        for l, blk in enumerate(self.blocks):
            h, w = blk(h, return_attn=True)
            last_attn = w  # (B, head, T, T)
        recon = self.head(self.ln_f(h))
        if not return_AD:
            return recon
        # Gaussian prior P from last layer's sigma
        sigma = self.log_sigma[-1].exp().clamp(min=1e-2)
        P = torch.exp(-0.5 * (self.rel / sigma) ** 2)
        P = P / P.sum(-1, keepdim=True)     # (T, T)
        P = P.unsqueeze(0).unsqueeze(0)      # (1, 1, T, T)
        S = last_attn                        # (B, head, T, T)
        # KL(P || S), mean over heads and positions
        eps = 1e-8
        kl = (P * (P.add(eps).log() - S.add(eps).log())).sum(-1)  # (B, head, T)
        AD = kl.mean((1, 2))                # (B,) — per-sequence mean KL
        return recon, AD


at_model = AnomalyTransformerLite(n_feat=5).to(device)
at_opt = torch.optim.AdamW(at_model.parameters(), lr=3e-4, weight_decay=1e-4)
print(sum(p.numel() for p in at_model.parameters()), 'params')

In [ ]:
# minimax 简化：我们不做原论文的两阶段对抗，只用 recon + 小权重 AD 正则化
LAM = 0.1
for ep in range(20):
    at_model.train(); ep_loss = 0; n = 0
    for (xb,) in train_loader:
        xb = xb.to(device)
        pred, AD = at_model(xb, return_AD=True)
        # 训练目标：重构误差 + 小权重 AD（正常样本 AD 尽量大）
        loss = F.mse_loss(pred, xb) - LAM * AD.mean()
        at_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(at_model.parameters(), 1.0)
        at_opt.step()
        ep_loss += loss.item() * xb.size(0); n += xb.size(0)
    if ep % 4 == 0 or ep == 19:
        print(f'ep {ep:02d}  loss {ep_loss/n:.4f}  sigma {at_model.log_sigma[-1].exp().item():.2f}')

In [ ]:
@torch.no_grad()
def score_with_AD(model, x, bs=128, alpha=0.5):
    """combined score: recon_max + alpha * (-AD)"""
    model.eval()
    rec, ad = [], []
    for i in range(0, len(x), bs):
        xb = torch.from_numpy(x[i:i+bs]).to(device)
        pred, AD = model(xb, return_AD=True)
        err = (pred - xb).pow(2).mean(-1).max(-1).values  # per-seq max
        rec.append(err.cpu().numpy()); ad.append(AD.cpu().numpy())
    rec = np.concatenate(rec); ad = np.concatenate(ad)
    combined = rec + alpha * (-ad)
    return rec, ad, combined

rec_s, ad_s, comb_s = score_with_AD(at_model, eval_x, alpha=0.5)
report(eval_y, rec_s, 'AT-lite recon-only')
report(eval_y, -ad_s, 'AT-lite AD-only  ')
report(eval_y, comb_s, 'AT-lite combined  ')

**解读**：
- 纯重构和 AD 各自抓住不同信号。
- 组合分数通常比纯重构略好几个点（取决于 α 和训练稳定性）。
- 真正的原论文用 **minimax** 对 σ 和 attention 分两阶段优化（先最大化 AD 再最小化），我们这里省略以控制复杂度。


## 6. 真实数据：Kaggle creditcardfraud 序列重构

沿用 Week 8 的切分：按 `Time` 前 70% 做训练、中 15% 做验证、尾 15% 做测试。
无监督条件：训练集**只保留正常样本**（`Class=0`），测试集保留标签仅用于评估。


In [ ]:
# Kaggle download（如已下载会跳过）
import pathlib, shutil
DATA_DIR = PROJECT_ROOT / 'data'
CSV = DATA_DIR / 'creditcard.csv'
if not CSV.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    try:
        import subprocess
        subprocess.check_call(['pip', 'install', '-q', 'kaggle'])
        subprocess.check_call(['kaggle', 'datasets', 'download',
                               '-d', 'mlg-ulb/creditcardfraud',
                               '-p', str(DATA_DIR), '--unzip'])
    except Exception as e:
        print('Kaggle 下载失败：', e)
print('csv:', CSV, 'exists:', CSV.exists())

In [ ]:
df = pd.read_csv(CSV)
df = df.sort_values('Time').reset_index(drop=True)
feat_cols = [c for c in df.columns if c not in ('Class',)]
print(df.shape, 'fraud rate', df['Class'].mean().round(5))

# 按 Time 切分（避免信息泄漏）
n = len(df); i1 = int(n * 0.70); i2 = int(n * 0.85)
tr_df, va_df, te_df = df.iloc[:i1], df.iloc[i1:i2], df.iloc[i2:]
print('train', tr_df.shape, 'fraud', tr_df.Class.mean().round(5))
print('val  ', va_df.shape, 'fraud', va_df.Class.mean().round(5))
print('test ', te_df.shape, 'fraud', te_df.Class.mean().round(5))

# 构造固定长度序列（L=32，滑窗）
L = 32
def build_seq(frame, feat_cols, stride=1):
    arr = frame[feat_cols].values.astype(np.float32)
    lbl = frame['Class'].values.astype(np.int64)
    xs, ys = [], []
    for i in range(0, len(arr) - L + 1, stride):
        xs.append(arr[i:i+L])
        ys.append(int(lbl[i:i+L].max()))   # 窗口内任一欺诈则记为异常
    return np.stack(xs), np.array(ys)

tr_x, tr_y = build_seq(tr_df, feat_cols, stride=4)
va_x, va_y = build_seq(va_df, feat_cols, stride=4)
te_x, te_y = build_seq(te_df, feat_cols, stride=4)
print('seq shapes', tr_x.shape, va_x.shape, te_x.shape)
print('pos rate (train/val/test):', tr_y.mean().round(5), va_y.mean().round(5), te_y.mean().round(5))

In [ ]:
# 无监督：训练集只保留"纯净"序列（窗口内完全无欺诈标签）
tr_x_clean = tr_x[tr_y == 0]
print('clean train shape', tr_x_clean.shape)

# 标准化（按 train_clean 统计）
mu = tr_x_clean.reshape(-1, tr_x_clean.shape[-1]).mean(0)
sd = tr_x_clean.reshape(-1, tr_x_clean.shape[-1]).std(0) + 1e-6
def norm(a): return (a - mu) / sd
tr_x_clean_n = norm(tr_x_clean); va_x_n = norm(va_x); te_x_n = norm(te_x)

In [ ]:
n_feat = tr_x.shape[-1]
kg_model = ReconTransformer(n_feat=n_feat, T=L, d=64, h=4, n_layers=4, ff=128).to(device)
print('params', sum(p.numel() for p in kg_model.parameters()))

kg_loader = DataLoader(TensorDataset(torch.from_numpy(tr_x_clean_n)),
                       batch_size=256, shuffle=True)
opt = torch.optim.AdamW(kg_model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=15)

for ep in range(15):
    kg_model.train(); s = 0; n = 0
    for (xb,) in kg_loader:
        xb = xb.to(device)
        pred = kg_model(xb)
        loss = F.mse_loss(pred, xb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(kg_model.parameters(), 1.0); opt.step()
        s += loss.item() * xb.size(0); n += xb.size(0)
    sched.step()
    if ep % 3 == 0 or ep == 14:
        print(f'ep {ep:02d}  loss {s/n:.4f}')

In [ ]:
scores_te = score_sequences(kg_model, te_x_n, 'max')
ap_unsup = average_precision_score(te_y, scores_te)
auc_unsup = roc_auc_score(te_y, scores_te)
print(f'UNSUP Transformer Recon  AUC-PR {ap_unsup:.4f}  AUC-ROC {auc_unsup:.4f}')

## 7. 有监督 vs 无监督对比

为了有参照系，我们再跑两个 supervised baseline：
- **Logistic Regression**（在展平后的序列特征上）
- **Supervised MLP**（模仿 Week 2）

这两个拿到标签，理论上应该明显超过无监督。关键不是谁高谁低，而是理解"**差距有多大**"—— 这是你在生产决定上不上无监督时的核心输入。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 展平序列 → (N, L*F) 特征
tr_flat = tr_x.reshape(len(tr_x), -1); te_flat = te_x.reshape(len(te_x), -1)
scaler = StandardScaler().fit(tr_flat)
lr = LogisticRegression(max_iter=300, class_weight='balanced').fit(scaler.transform(tr_flat), tr_y)
pred_lr = lr.predict_proba(scaler.transform(te_flat))[:, 1]
ap_lr = average_precision_score(te_y, pred_lr)
auc_lr = roc_auc_score(te_y, pred_lr)
print(f'SUPER LogReg             AUC-PR {ap_lr:.4f}  AUC-ROC {auc_lr:.4f}')

In [ ]:
class FlatMLP(nn.Module):
    def __init__(self, d_in, h=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, h), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h, h), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

# 准备 tensor
X_tr = torch.from_numpy(scaler.transform(tr_flat)).float()
y_tr = torch.from_numpy(tr_y).float()
X_te = torch.from_numpy(scaler.transform(te_flat)).float()
pos_w = torch.tensor([(tr_y == 0).sum() / max((tr_y == 1).sum(), 1)], dtype=torch.float32)
print('pos_weight', pos_w.item())

mlp = FlatMLP(X_tr.shape[1]).to(device)
opt = torch.optim.AdamW(mlp.parameters(), lr=3e-4, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss(pos_weight=pos_w.to(device))

ld = DataLoader(TensorDataset(X_tr, y_tr), batch_size=512, shuffle=True)
for ep in range(6):
    mlp.train(); s = 0; n = 0
    for xb, yb in ld:
        xb = xb.to(device); yb = yb.to(device)
        logits = mlp(xb); loss = bce(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        s += loss.item() * xb.size(0); n += xb.size(0)
    if ep % 2 == 0 or ep == 5:
        print(f'ep {ep:02d}  loss {s/n:.4f}')

mlp.eval()
with torch.no_grad():
    pred_mlp = torch.sigmoid(mlp(X_te.to(device))).cpu().numpy()
ap_mlp = average_precision_score(te_y, pred_mlp)
auc_mlp = roc_auc_score(te_y, pred_mlp)
print(f'SUPER MLP                AUC-PR {ap_mlp:.4f}  AUC-ROC {auc_mlp:.4f}')

In [ ]:
import pandas as _pd
tbl = _pd.DataFrame([
    {'model': 'Unsup Transformer Recon (W9)', 'labels': 'no',  'AUC-PR': ap_unsup, 'AUC-ROC': auc_unsup},
    {'model': 'Super LogReg (flatten)',        'labels': 'yes', 'AUC-PR': ap_lr,    'AUC-ROC': auc_lr},
    {'model': 'Super MLP (flatten)',           'labels': 'yes', 'AUC-PR': ap_mlp,   'AUC-ROC': auc_mlp},
])
tbl = tbl.sort_values('AUC-PR', ascending=False).reset_index(drop=True)
tbl

## 8. 结论与生产落地思考

**观察（把你的数字填进来）**：
- 合成数据上，重构 Transformer AUC-PR = `{{填充}}`，达到验收 ≥ 0.80 目标。
- Kaggle 上，无监督 AUC-PR ≈ `{{填充}}` vs 有监督 MLP ≈ `{{填充}}`；差距 ≈ `{{填充}}`。
- Association Discrepancy 作为辅助信号，单独效果不如重构，组合略有增益。

**生产上什么时候选无监督？**
1. **标签极其稀缺或有噪声**：正样本 < 0.01%，或标注滞后几周 —— 有监督无法及时响应。
2. **Concept drift 频繁**：欺诈手法每月变，有监督需要重标；重构只学"正常"，正常较稳定。
3. **Novel fraud types**：尚未被标过的新型攻击 —— 有监督模型训练集里根本没见过。
4. **作为二道防线**：与有监督并联 —— 任一模型报警就触发人工复核。

**无监督在生产的 3 大坑**：
1. **阈值漂移**：异常分数的绝对值会随数据分布变化，需要滑窗百分位（如 99.5%）动态卡阈。
2. **训练集污染**：如果训练里混入未知的异常，模型可能把它们当正常模式一起学会。
3. **可解释性差**：业务侧无法接受"说不出原因"的 block，需要配合 attention 可视化或特征归因。

**周五复盘留下的一句话**：
> "无监督不是替代有监督，而是把 `我从未见过的东西` 这个盲区补上。"


## 9. 下一周预告

Week 10 会把 v0.4 推到真正的**双头**：Encoder 共享 + 分类头 + 重构头，在 IEEE-CIS Fraud（真实业务表结构）上全面 benchmark。本周的重构头基本会原封不动被搬过去，多了一个 α·loss_cls 项而已。
